In [ ]:
from pathlib import Path
import re

import pandas as pd
from lxml import etree


In [ ]:
data_dir = Path('../data/xml')
xml_files = sorted(data_dir.glob('*.xml'))
len(xml_files), [p.name for p in xml_files]

In [ ]:
parser = etree.XMLParser(recover=True, huge_tree=True)

test_doc = etree.parse(str(xml_files[0]), parser)
root = test_doc.getroot()

root.tag, root.nsmap

In [ ]:
def get_tei_ns(root) -> dict:
    """Gibt einen Namespace-Mapper für TEI zurück (falls Default-Namespace gesetzt ist)."""
    ns = root.nsmap.get(None)
    return {'tei': ns} if ns else {}

ns = get_tei_ns(root)
ns

In [ ]:
letter_divs = root.xpath(".//tei:div[@subtype='letter']", namespaces=ns) if ns else root.xpath(".//*[@subtype='letter' and local-name()='div']")
len(letter_divs)

In [ ]:
one = letter_divs[1]  # --> wir schauen uns den zweiten Brief an

# Briefnummer
one.get('n')

In [ ]:
# Buchdiv als Vorfahr suchen
book_div = one.xpath("ancestor::tei:div[@subtype='Book'][1]", namespaces=ns) if ns else one.xpath("ancestor::*[local-name()='div' and @subtype='Book'][1]")
book_n = book_div[0].get('n') if book_div else None
book_n

In [ ]:
def norm_ws(s: str) -> str:
    return re.sub(r'\s+', ' ', s).strip()

def text_of_first(nodes):
    if not nodes:
        return None
    return norm_ws(' '.join(nodes[0].itertext()))

dateline = text_of_first(one.xpath(".//tei:seg[@rend='dateline']", namespaces=ns)) if ns else text_of_first(one.xpath(".//*[local-name()='seg' and @rend='dateline']"))
salute = text_of_first(one.xpath(".//tei:seg[@rend='salute']", namespaces=ns)) if ns else text_of_first(one.xpath(".//*[local-name()='seg' and @rend='salute']"))
date_el = one.xpath(".//tei:date[1]", namespaces=ns) if ns else one.xpath(".//*[local-name()='date'][1]")
date_when = date_el[0].get('when') if date_el else None

dateline, salute, date_when

In [ ]:
def extract_sender_recipient_from_salute(salute):
    """Heuristik für:
    - 'Cicero Bruto salutem' -> ('Cicero', 'Bruto')
    - 'M. Cicero S.D. P. Lentulo procos.' -> ('M. Cicero', 'P. Lentulo procos')
    """
    if not salute:
        return None, None

    s = norm_ws(str(salute)).replace('"', '').replace("'", '')
    # Schlusswort salutem am Ende entfernen
    s = re.sub(r'\b(salutem)\b\.?\s*$', '', s, flags=re.IGNORECASE).strip()

    # Trenner S.D. (salutem dicit)
    m = re.search(r'\bS\s*\.?\s*D\s*\.?\b', s, flags=re.IGNORECASE)
    if m:
        sender = norm_ws(s[:m.start()]) or None
        recipient = norm_ws(s[m.end():])
        recipient = re.sub(r'^[\s\.,;:]+', '', recipient)
        recipient = re.sub(r'[\s\.,;:]+$', '', recipient)
        recipient = recipient or None
        return sender, recipient

    parts = s.split()
    if len(parts) < 2:
        return None, None
    return parts[0], parts[1]

extract_sender_recipient_from_salute(salute)

In [ ]:
def parse_year(date_when):
    if not date_when:
        return None
    s = str(date_when).strip()
    return int(s) if re.match(r'^-?\d{1,4}$', s) else None

print(parse_year(date_when))

In [ ]:
sender, recipient = extract_sender_recipient_from_salute(salute)

row = {
    'corpus': 'TEST',
    'book_n': book_n,
    'letter_n': one.get('n'),
    'date_when': date_when,
    'year': parse_year(date_when),
    'dateline': dateline,
    'salute': salute,
    'sender': sender,
    'recipient': recipient,
}

row

In [ ]:
def corpus_label_from_filename(p: Path) -> str:
    name = p.name.lower()
    if 'ad-familiares' in name:
        return 'ad_familiares'
    if 'ad-atticum' in name:
        return 'ad_atticum'
    if 'ad-quintum' in name:
        return 'ad_quintum_fratrem'
    if 'ad-brutum' in name:
        return 'ad_m_brutum'
    return p.stem

rows = []

for path in xml_files:
    corpus = corpus_label_from_filename(path)
    doc = etree.parse(str(path), parser)
    root = doc.getroot()
    ns = get_tei_ns(root)

    letter_divs = root.xpath(".//tei:div[@subtype='letter']", namespaces=ns) if ns else root.xpath(".//*[@subtype='letter' and local-name()='div']")

    for ld in letter_divs:
        book_div = ld.xpath("ancestor::tei:div[@subtype='Book'][1]", namespaces=ns) if ns else ld.xpath("ancestor::*[local-name()='div' and @subtype='Book'][1]")
        book_n = book_div[0].get('n') if book_div else None
        letter_n = ld.get('n')

        dateline = text_of_first(ld.xpath(".//tei:seg[@rend='dateline']", namespaces=ns)) if ns else text_of_first(ld.xpath(".//*[local-name()='seg' and @rend='dateline']"))
        salute = text_of_first(ld.xpath(".//tei:seg[@rend='salute']", namespaces=ns)) if ns else text_of_first(ld.xpath(".//*[local-name()='seg' and @rend='salute']"))
        date_el = ld.xpath(".//tei:date[1]", namespaces=ns) if ns else ld.xpath(".//*[local-name()='date'][1]")
        date_when = date_el[0].get('when') if date_el else None

        sender, recipient = extract_sender_recipient_from_salute(salute)

        #für text:
        paragraphs = ld.xpath(".//tei:p", namespaces=ns) if ns else ld.xpath(".//*[local-name()='p']")
        text = " ".join(" ".join(t.strip() for t in p.itertext() if t and t.strip()) for p in paragraphs)
        text = re.sub(r"\s+", " ", text).strip()

        rows.append({
            'corpus': corpus,
            'book_n': book_n,
            'letter_n': letter_n,
            'date_when': date_when,
            'year': parse_year(date_when),
            'dateline': dateline,
            'salute': salute,
            'sender': sender,
            'recipient': recipient,
            'text': text
        })

len(rows)

In [ ]:
df = pd.DataFrame(rows)


In [ ]:
probleme = df[df['recipient'].isna()][['salute', 'sender']]

In [ ]:
def korrekturen_anwenden(row):
    if row['salute'] in korrekturen:
        row['sender'], row['recipient'] = korrekturen[row['salute']]
    return row

In [ ]:
korrekturen = {
    "CICERO APPIO IMP. S. D.": ("CICERO", "APPIO IMP."),
    "CICERO APPIO PVLCHRO VT SPERO, CENSORI S. D.": ("CICERO", "APPIO PVLCHRO"),
    "M. TVLLIVS M. F. CICERO Q. METELLO Q. F. CELERI PROCOS. S. D.": ("CICERO", "Q. METELLO Q.F.CELERI"),
    "CICERO CAESARI IMP. S. D.":("CICERO", "CAESARI IMP."),
    "CICERO CVRIO S. D." : ("CICERO","CVRIO"),
    "CICERO PAETO S. D." : ("CICERO", "PAETO"),
    "C. ASINIVS POLLIO CICERONI S. D.": ("C. ASINIVS POLLIO", "CICERONI"),
    "D. BRVTVS COS. DESIG. M. CICERONI S. D.": ("D. BRVTVS COS. DESIG.", "M.CICERONI"),
    "M. CICERO D. BRVTO COS. DESIG. S. D.": ("M. CICERO", "D. BRVTO COS. DESIG."),
    "M. CICERO D. BRVTO COS. DES. S. D.": ("M. CICERO", "D. BRVTO COS. DES."),
    "M. CICERO D. BRVTO S. D.": ("M. CICERO", "D. BRVTO"),
    "D. BRVTO COS. DESIG." : ("CICERO", "OPPIO"),
    "CICERO C. SEXTILIO RVFO QVAESTORI S. D.": ("CICERO", "C. SEXTILIO RVFO QVAESTORI" ),
    "CICERO P. CAESIO S. D.": ("CICERO","P.CAESIO"),
    "M. CICERO T. TITIO T. F. LEG. S. D.": ("M. CICERO", "T. TITIO T. F. LEG."),
    "M. CICERO IIII VIRIS ET DECVRIONIBVS S. D." : ("M. CICERO", "IIII VIRIS ET DECVRIONIBVS"),
    "TVLLIVS TERENTIAE SVAE, TVLLIOLAE SVAE, CICERONI SVO S. D.": ("TVLLIVS","TERENTIAE SVAE, TVLLIOLAE SVAE, CICERONI SVO"),
    "TVLLIVS TERENTIAE SVAE S. D.": ("TVLLIVS", "TERENTIAE SVAE"),
    "Q. CICERO TIRONI S. D." : ("Q.CICERO", "TIRONI"),
    "QVINTVS TIRONI SV0 P. S. D.": ("QVINTVS", "TIRONI SVO P."),
    "CICERO ATTICO S. D.": ("CICERO", "ATTICO"),
    "CICERO ANTONIO COS. S. D.": ("CICERO", "ANTONIO COS.")
    }

In [ ]:

df = df.apply(korrekturen_anwenden, axis=1)

In [ ]:
def Namenssplitting_besser(row):
    if pd.notna(row['sender']) and pd.notna(row['recipient']):
        if re.fullmatch(r'[A-Z]\.', row['sender']) and row['recipient'] == 'CICERO':
            parts = row['salute'].split("CICERO", 1)
            if len(parts) == 2:
                row['sender'] = parts[0].strip() + " CICERO"
                row['recipient'] = parts[1].strip()
                row['recipient'] = re.sub(r'\bS\..*$', '', row['recipient']).strip()
    return row

In [ ]:

df = df.apply(Namenssplitting_besser, axis=1)

In [ ]:
def fix_teremia(row):
    if pd.notna(row['sender']):
        if row['sender'] == 'TVLLIVS TEREMIAE SVAE':

            row['sender'] = 'TVLLIVS'
            row['recipient'] = 'TEREMIAE'

    return row

In [ ]:

df = df.apply(fix_teremia, axis=1)

In [ ]:
def mehrere_absender(row):
    if pd.notna(row['sender']):
        if row['sender'] == 'TVLLIVS ET CICERO, TERENTIA, TVLLIA Q. Q. TIRONI S. P. D.':

            row['sender'] = 'TVLLIVS ET CICERO, TERENTIA, TVLLIA'
            row['recipient'] = 'TIRONI'

    return row

In [ ]:
df = df.apply(mehrere_absender, axis=1)

In [ ]:
def Daten_besser_matchen(row):
    form = r'\((\d+)\)'
    match = re.search(form, row['dateline'])
    if pd.isna(row['date_when']):
        if match:
            row['date_when'] = match.group(1) 
    return row

In [ ]:
df = df.apply(Daten_besser_matchen, axis=1)


In [ ]:
def jahreszahlen_aufbereiten(row):
    if pd.isna(row['date_when']):
        return row
    if re.search(r'-00\d\d', row['date_when']):
        row['year'] = int(row['date_when'])
    else:
        row['year'] = -int(row['date_when'])
    return row
    


In [ ]:
df = df.apply(jahreszahlen_aufbereiten, axis=1)


In [ ]:
out_dir = Path('../data')
out_dir.mkdir(exist_ok=True)

mini_cols = ['corpus','book_n','letter_n','sender','recipient','date_when','year','dateline', 'text']
mini = df[mini_cols].copy()

mini_path = out_dir / 'cicero_letters.csv'
mini.to_csv(mini_path, index=False)
mini_path